In [1]:
# ============================================================
# EVALUAR bw.wav COMO POSIBLE IMPOSTOR
# Modelos: ECAPA y WAVLM
# Protocolos: A y B
# ============================================================

import pickle
import numpy as np
import pandas as pd
import torch
import soundfile as sf

from pathlib import Path
from scipy.signal import resample_poly
from sklearn.metrics.pairwise import cosine_similarity

from speechbrain.inference.speaker import EncoderClassifier
from transformers import AutoFeatureExtractor, WavLMForXVector


# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BW_PATH = Path("bw.wav")
RESULTS_DIR = Path("resultados")
PROTOCOL_DIR = Path("protocolo_vox1")

MODELOS = ["ECAPA", "WAVLM"]

print("Dispositivo:", DEVICE)

# ------------------------------------------------------------
# 2. CARGAR EMBEDDINGS, PROTOCOLOS Y UMBRALES
# ------------------------------------------------------------

with open("emb_ecapa.pkl", "rb") as f:
    emb_ecapa = pickle.load(f)

with open("emb_wavlm.pkl", "rb") as f:
    emb_wavlm = pickle.load(f)

EMBEDDINGS = {
    "ECAPA": emb_ecapa,
    "WAVLM": emb_wavlm
}

# Protocolo A
protocolo_A = pd.read_csv(
    PROTOCOL_DIR / "veri_test_original.txt",
    sep=r"\s+",
    header=None,
    names=["label", "audio1", "audio2"]
)

# Protocolo B
enrollment_B = pd.read_csv(
    PROTOCOL_DIR / "enrollment_10.csv"
)

# Umbrales EER
metricas = pd.read_csv(RESULTS_DIR / "metricas.csv")

UMBRALES = {
    fila["modelo_protocolo"]: fila["umbral_EER"]
    for _, fila in metricas.iterrows()
}

print("\nUmbrales:")
for nombre, valor in UMBRALES.items():
    print(f"{nombre}: {valor:.6f}")


# ------------------------------------------------------------
# 3. FUNCIONES AUXILIARES
# ------------------------------------------------------------

def normalizar_ruta(ruta):
    return str(ruta).replace("\\", "/").lstrip("./")


def buscar_embedding(ruta, embeddings):
    """
    Busca un embedding por coincidencia exacta o por sufijo.
    """
    ruta = normalizar_ruta(ruta)

    if ruta in embeddings:
        return embeddings[ruta]

    coincidencias = [
        k for k in embeddings
        if normalizar_ruta(k).endswith(ruta)
    ]

    if len(coincidencias) == 1:
        return embeddings[coincidencias[0]]

    if len(coincidencias) > 1:
        raise ValueError(
            f"Ruta ambigua: {ruta}\n"
            f"Coincidencias: {coincidencias[:5]}"
        )

    raise KeyError(f"No se encontró embedding para: {ruta}")


def normalizar_embedding(embedding):
    embedding = np.asarray(embedding, dtype=np.float32).reshape(-1)
    norma = np.linalg.norm(embedding)

    if norma == 0:
        raise ValueError("Embedding con norma cero")

    return embedding / norma


def similitud_coseno(a, b):
    a = normalizar_embedding(a)
    b = normalizar_embedding(b)

    return float(np.dot(a, b))


def cargar_audio_16k(ruta):
    audio, sr = sf.read(ruta)

    # Convertir a mono
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)

    audio = audio.astype(np.float32)

    # Remuestrear a 16 kHz
    if sr != 16000:
        audio = resample_poly(audio, 16000, sr).astype(np.float32)

    return audio


# ------------------------------------------------------------
# 4. CARGAR MODELOS Y EXTRAER EMBEDDING DE bw.wav
# ------------------------------------------------------------

if not BW_PATH.exists():
    raise FileNotFoundError(f"No se encontró {BW_PATH.resolve()}")

audio_bw = cargar_audio_16k(BW_PATH)

print("\nDuración bw.wav:", len(audio_bw) / 16000, "segundos")

# ECAPA
print("\nCargando ECAPA...")

ecapa_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    run_opts={"device": DEVICE}
)

waveform = torch.tensor(
    audio_bw,
    dtype=torch.float32
).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    embedding_bw_ecapa = ecapa_model.encode_batch(waveform)

embedding_bw_ecapa = (
    embedding_bw_ecapa.squeeze().cpu().numpy()
)

embedding_bw_ecapa = normalizar_embedding(embedding_bw_ecapa)


# WAVLM
print("Cargando WavLM...")

wavlm_name = "microsoft/wavlm-base-plus-sv"

extractor = AutoFeatureExtractor.from_pretrained(wavlm_name)

wavlm_model = WavLMForXVector.from_pretrained(
    wavlm_name
).to(DEVICE)

inputs = extractor(
    audio_bw,
    sampling_rate=16000,
    return_tensors="pt"
)

inputs = {
    k: v.to(DEVICE)
    for k, v in inputs.items()
}

with torch.no_grad():
    output = wavlm_model(**inputs)

embedding_bw_wavlm = output.embeddings.squeeze().cpu().numpy()
embedding_bw_wavlm = normalizar_embedding(embedding_bw_wavlm)


EMBEDDINGS_BW = {
    "ECAPA": embedding_bw_ecapa,
    "WAVLM": embedding_bw_wavlm
}

print("\nEmbeddings de bw.wav extraídos correctamente.")


audios_A = pd.unique(
    pd.concat([
        protocolo_A["audio1"],
        protocolo_A["audio2"]
    ])
)

resultados_A = {}

for modelo in MODELOS:

    print(f"\nEvaluando Protocolo A - {modelo}...")

    embeddings = EMBEDDINGS[modelo]
    embedding_bw = EMBEDDINGS_BW[modelo]

    umbral = UMBRALES[f"{modelo} - A"]

    resultados = []

    for audio_ref in audios_A:

        embedding_ref = buscar_embedding(
            audio_ref,
            embeddings
        )

        score = similitud_coseno(
            embedding_bw,
            embedding_ref
        )

        # Score >= umbral: aceptado como mismo hablante
        aceptado = score >= umbral

        resultados.append({
            "audio_referencia": audio_ref,
            "score": score,
            "aceptado": aceptado
        })

    df = pd.DataFrame(resultados)
    df = df.sort_values("score", ascending=False)

    resultados_A[modelo] = df

    print("Umbral:", round(umbral, 6))
    print("Comparaciones:", len(df))
    print("Aceptaciones:", df["aceptado"].sum())
    print("Score máximo:", round(df["score"].max(), 6))



resultados_B = {}

for modelo in MODELOS:

    print(f"\nEvaluando Protocolo B - {modelo}...")

    embeddings = EMBEDDINGS[modelo]
    embedding_bw = EMBEDDINGS_BW[modelo]

    umbral = UMBRALES[f"{modelo} - B"]

    resultados = []

    # Construir un template por hablante
    templates = {}

    for speaker_id, grupo in enrollment_B.groupby("speaker_id"):

        embeddings_speaker = []

        for _, fila in grupo.iterrows():

            embedding = buscar_embedding(
                fila["audio_path"],
                embeddings
            )

            embeddings_speaker.append(
                normalizar_embedding(embedding)
            )

        # Promedio de los embeddings de enrollment
        template = np.mean(
            embeddings_speaker,
            axis=0
        )

        template = normalizar_embedding(template)

        templates[speaker_id] = template

    # Comparar bw.wav contra cada hablante
    for speaker_id, template in templates.items():

        score = similitud_coseno(
            embedding_bw,
            template
        )

        aceptado = score >= umbral

        resultados.append({
            "speaker_id": speaker_id,
            "score": score,
            "aceptado": aceptado
        })

    df = pd.DataFrame(resultados)
    df = df.sort_values("score", ascending=False)

    resultados_B[modelo] = df

    print("Umbral:", round(umbral, 6))
    print("Hablantes evaluados:", len(df))
    print("Aceptaciones:", df["aceptado"].sum())
    print("Score máximo:", round(df["score"].max(), 6))



resumen = []

for modelo in MODELOS:

    for protocolo, resultados in [
        ("A", resultados_A[modelo]),
        ("B", resultados_B[modelo])
    ]:

        umbral = UMBRALES[f"{modelo} - {protocolo}"]

        score_max = resultados["score"].max()

        n_aceptaciones = resultados["aceptado"].sum()

        es_impostor = n_aceptaciones == 0

        resumen.append({
            "modelo": modelo,
            "protocolo": protocolo,
            "umbral_EER": umbral,
            "score_maximo": score_max,
            "aceptaciones": n_aceptaciones,
            "total_comparaciones": len(resultados),
            "resultado": (
                "IMPOSTOR (rechazado por todos)"
                if es_impostor
                else "ACEPTADO por al menos una referencia"
            )
        })


resumen = pd.DataFrame(resumen)

print("RESULTADOS FINALES PARA bw.wav")

display(
    resumen.style.format({
        "umbral_EER": "{:.6f}",
        "score_maximo": "{:.6f}"
    })
)



for modelo in MODELOS:

    print(f"{modelo} - PROTOCOLO A: TOP 10")

    display(resultados_A[modelo].head(10))

    print(f"\n{modelo} - PROTOCOLO B: TOP 10")

    display(resultados_B[modelo].head(10))


RESULTS_DIR.mkdir(exist_ok=True)

with open(RESULTS_DIR / "evaluacion_bw.pkl", "wb") as f:
    pickle.dump({
        "resumen": resumen,
        "resultados_A": resultados_A,
        "resultados_B": resultados_B
    }, f)

resumen.to_csv(
    RESULTS_DIR / "resumen_bw.csv",
    index=False
)

for modelo in MODELOS:

    resultados_A[modelo].to_csv(
        RESULTS_DIR / f"bw_{modelo}_A.csv",
        index=False
    )

    resultados_B[modelo].to_csv(
        RESULTS_DIR / f"bw_{modelo}_B.csv",
        index=False
    )

print("\nResultados guardados en:", RESULTS_DIR.resolve())

Dispositivo: cpu

Umbrales:
ECAPA - A: 0.272333
ECAPA - B: 0.371500
WAVLM - A: 0.873540
WAVLM - B: 0.908500

Duración bw.wav: 3.4113125 segundos

Cargando ECAPA...


Cargando WavLM...


Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

c:\Users\benja\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\functional.py:6829: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(



Embeddings de bw.wav extraídos correctamente.

Evaluando Protocolo A - ECAPA...
Umbral: 0.272333
Comparaciones: 4715
Aceptaciones: 22
Score máximo: 0.390125

Evaluando Protocolo A - WAVLM...
Umbral: 0.87354
Comparaciones: 4715
Aceptaciones: 336
Score máximo: 0.947298

Evaluando Protocolo B - ECAPA...
Umbral: 0.3715
Hablantes evaluados: 40
Aceptaciones: 0
Score máximo: 0.216012

Evaluando Protocolo B - WAVLM...
Umbral: 0.9085
Hablantes evaluados: 40
Aceptaciones: 1
Score máximo: 0.930297
RESULTADOS FINALES PARA bw.wav


,modelo,protocolo,umbral_EER,score_maximo,aceptaciones,total_comparaciones,resultado
0,ECAPA,A,0.272333,0.390125,22,4715,ACEPTADO por al menos una referencia
1,ECAPA,B,0.371500,0.216012,0,40,IMPOSTOR (rechazado por todos)
2,WAVLM,A,0.873540,0.947298,336,4715,ACEPTADO por al menos una referencia
3,WAVLM,B,0.908500,0.930297,1,40,ACEPTADO por al menos una referencia


... (5 outputs left out)

,audio_referencia,score,aceptado
2500,id10292/v6MWr5UAZ94/00006.wav,0.947298,True
2541,id10292/6Bh6P9nGsM4/00001.wav,0.946461,True
820,id10277/cELY3LdIo-0/00003.wav,0.938819,True
819,id10277/cELY3LdIo-0/00002.wav,0.937733,True
2425,id10292/N8_clYCM9n8/00002.wav,0.937067,True
4066,id10305/Piw0wvQbpnQ/00012.wav,0.936385,True
2657,id10293/k1jiAINXHRI/00002.wav,0.934799,True
814,id10277/UIbPolhxLDI/00001.wav,0.933952,True
4085,id10305/cpXXdMJuCdw/00001.wav,0.933924,True
2501,id10292/v6MWr5UAZ94/00007.wav,0.933917,True



WAVLM - PROTOCOLO B: TOP 10


,speaker_id,score,aceptado
7,id10277,0.930297,True
35,id10305,0.901054,False
22,id10292,0.891496,False
23,id10293,0.839250,False
1,id10271,0.780094,False
2,id10272,0.779496,False
6,id10276,0.776890,False
3,id10273,0.750781,False
32,id10302,0.749592,False
9,id10279,0.738410,False



Resultados guardados en: C:\Users\benja\OneDrive\Desktop\Universidad\Tareas\Voz\tarea3-verificacion-locutor\resultados
